In [1]:
# Setup

In [ ]:
!pip install GPUtil

import torch
from GPUtil import showUtilization as gpu_usage
from numba import cuda

def free_gpu_cache():
    print("Initial GPU Usage")
    gpu_usage()                             

    torch.cuda.empty_cache()

    cuda.select_device(0)
    cuda.close()
    cuda.select_device(0)

    print("GPU Usage after emptying the cache")
    gpu_usage()

free_gpu_cache()                           


In [3]:
!nvidia-smi

Fri Feb 21 21:06:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             25W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import cv2
from sklearn.model_selection import StratifiedKFold
import numpy as np
from tqdm import tqdm

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.4 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


# Imports

In [5]:
# Read in the training dataset
train = pd.read_csv('/kaggle/input/lacuna-solar-survey-challenge/Train.csv')

In [6]:
train

,ID,img_origin,placement,boil_nbr,pan_nbr,polygon
0,ID00rw8,D,roof,0,2,"[(2087, 2179.0), (2181, 2191.0), (2171, 2223.0..."
1,ID014O6EC7,D,roof,0,1,"[(1327, 1574.0), (1595, 1308.0), (2169, 1744.0..."
2,ID020cu0z,D,openspace,0,1,"[(2215, 1372.0), (2400, 1422.0), (2316, 1716.0..."
3,ID024YTBkLvRpQahT,D,roof,0,1,"[(1574, 526.0), (1698, 366.0), (1810, 422.0), ..."
4,ID024YTBkLvRpQahT,D,roof,0,2,"[(1402, 782.0), (1814, 854.0), (1726, 1074.0),..."
...,...,...,...,...,...,...
6786,IDzt0z6lQC,S,S-unknown,1,0,"[(277, 186.0), (326, 215.0), (306, 262.0), (24..."
6787,IDzwqGWYJzj,D,roof,0,2,"[(2027, 1928.0), (2226, 1766.0), (2548, 2052.0..."
6788,IDzxHMiMmh9r04J,D,openspace,0,2,"[(1257, 1267.0), (1344, 1169.0), (1711, 1463.0..."
6789,IDzxXgqh,D,roof,0,1,"[(3448, 2606.0), (3227, 2420.0), (3662, 2208.0..."


In [7]:
train["boil_nbr"].unique()

array([0, 1, 2, 3, 4, 5])

In [8]:
train["pan_nbr"].unique()

array([  2,   1,  29,   0,   7,   3,  18,   4,   6,  12,  21,  23,  22,
         5,   8,  10,  16,   9,  28,  20,  11,  14,  51,  40,  32,  27,
        24,  81,  60,  66,  19,  15,  54, 161,  31,  36, 124, 100,  30,
        33, 108,  75,  46,  88, 130,  90,  13,  34,  72, 127, 120, 109,
       138, 184, 173,  25,  38,  56,  52,  49,  96,  35, 152,  50])

In [9]:
train["placement"].unique()

array(['roof', 'openspace', 'S-unknown', 'r_openspace'], dtype=object)

In [10]:
(train['placement'] == 'S-unknown').sum()

1350

In [11]:
# Create a placement mapper
placement_mapper = train[["ID", "placement"]].drop_duplicates().set_index("ID").to_dict()
# Create a "img_origin" mapper
img_origin_mapper = train[["ID", "img_origin"]].drop_duplicates().set_index("ID").to_dict()

# Group by "ID" and sum up boil_nb, pan_nbr
train_df = train.groupby("ID").sum().reset_index()[["ID", "boil_nbr", "pan_nbr"]]

# Map img_origin and placement
train_df["img_origin"] = train_df["ID"].map(img_origin_mapper["img_origin"])
train_df["placement"] = train_df["ID"].map(placement_mapper["placement"])

# Create path column
train_df["path"] = "/kaggle/input/lacuna-solar-survey-challenge/images/" + train_df["ID"] + ".jpg"

In [12]:
train_df.head()

,ID,boil_nbr,pan_nbr,img_origin,placement,path
0,ID00rw8,0,2,D,roof,/kaggle/input/lacuna-solar-survey-challenge/im...
1,ID014O6EC7,0,1,D,roof,/kaggle/input/lacuna-solar-survey-challenge/im...
2,ID020cu0z,0,1,D,openspace,/kaggle/input/lacuna-solar-survey-challenge/im...
3,ID024YTBkLvRpQahT,0,5,D,roof,/kaggle/input/lacuna-solar-survey-challenge/im...
4,ID02vByTw8Htl,0,1,D,roof,/kaggle/input/lacuna-solar-survey-challenge/im...


In [13]:
def adjust_contrast(image):
    # Convert to YUV color space
    yuv = cv2.cvtColor(image, cv2.COLOR_RGB2YUV)
    # Apply histogram equalization to the Y channel (luminance)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    # Convert back to RGB
    image = cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)
    return image

def detect_edges(image, low_threshold=100, high_threshold=200):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low_threshold, high_threshold)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    
def sharpen_image(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.5)
    sharpened = cv2.addWeighted(image, 1.5, blurred, -0.5, 0)
    return sharpened

In [14]:
# # Stratified KFold based on multi-label targets
# train_df["stratify_label"] = train_df[["boil_nbr", "pan_nbr"]].sum(axis=1)
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# train_df["fold"] = -1
# for fold, (_, valid_idx) in enumerate(skf.split(train_df, train_df["stratify_label"])):
#     train_df.loc[valid_idx, "fold"] = fold

In [15]:
# Define Transformations
train_transforms = A.Compose([
    A.RandomResizedCrop(384, 384, scale=(0.8, 1.0), p=1.0),  # Random crop + resize to improve generalization
    A.HorizontalFlip(p=0.5),  # Horizontal flip with 50% probability
    A.RandomBrightnessContrast(p=0.2),  # Random brightness and contrast adjustment
    A.HueSaturationValue(p=0.3),  # Random adjustment of color saturation, hue, and value
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),  # Shift, scale, rotate
    A.GridDistortion(p=0.2),  # Grid distortion for improved robustness
    A.ElasticTransform(p=0.2),  # Elastic transform for spatial variation
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.2),  # CoarseDropout as an alternative to Cutout
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Standard normalization
    ToTensorV2()
])


test_transforms = A.Compose([
    A.Resize(384, 384),  # Resize the image for consistency
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Standard normalization
    ToTensorV2()
])


In [16]:
class SolarPanelDataset(Dataset):
    def __init__(self, dataframe, transform=None, to_train=True):
        self.dataframe = dataframe
        self.transform = transform
        self.to_train = to_train

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = cv2.imread(row["path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # # Apply additional preprocessing steps
        # image = sharpen_image(image)  # Apply sharpening
        # image = adjust_contrast(image)  # Apply contrast adjustment
        # image = detect_edges(image)  # Apply edge detection

        # Apply the transformations (augmentation for training)
        if self.transform:
            image = self.transform(image=image)["image"]

        # If training, return the target labels (boil_nbr, pan_nbr)
        if self.to_train:
            target = torch.tensor([row["boil_nbr"], row["pan_nbr"]], dtype=torch.float32)
            return image, target
        else:
            return image


In [17]:
train_df["stratify_label"] = train_df[["boil_nbr", "pan_nbr"]].sum(axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_df["fold"] = -1
for fold, (_, valid_idx) in enumerate(skf.split(train_df, train_df["stratify_label"])):
    train_df.loc[valid_idx, "fold"] = fold

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [18]:
# Prepare Dataloaders
fold = 0  # Change fold index as needed
train_data = train_df[train_df["fold"] != fold].reset_index(drop=True)
valid_data = train_df[train_df["fold"] == fold].reset_index(drop=True)

dataset_train = SolarPanelDataset(train_data, transform=train_transforms)
dataset_valid = SolarPanelDataset(valid_data, transform=test_transforms)

train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=os.cpu_count())
valid_loader = DataLoader(dataset_valid, batch_size=32, shuffle=False) 

In [19]:
# Model Definition
class EfficientNetRegressor(nn.Module):
    def __init__(self):
        super(EfficientNetRegressor, self).__init__()
        self.model = timm.create_model("tf_efficientnet_b4", pretrained=True)
        self.model.classifier = nn.Linear(self.model.classifier.in_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection import backbones
from torchvision.models.detection.rpn import AnchorGenerator

class ResNetRCNNRegressor(nn.Module):
    def __init__(self):
        super(ResNetRCNNRegressor, self).__init__()

        # Load the pre-trained ResNet backbone using torchvision's resnet_fpn_backbone
        backbone = backbones.resnet_fpn_backbone('resnet50', pretrained=True)
        
        # Create a Faster R-CNN model using the ResNet backbone
        self.model = FasterRCNN(
            backbone,
            num_classes=2,  # Adjust number of classes for regression task (typically 2 for coordinates or other tasks)
            rpn_anchor_generator=AnchorGenerator(
                sizes=((32, 64, 128, 256, 512),) * 5,
                aspect_ratios=((0.5, 1.0, 2.0) * 5,)
            ),
            box_detections_per_img=100,  # Number of detections per image
            box_score_thresh=0.5  # Threshold for filtering the detections
        )
        
        # Modify the output layer for your task, assuming you want a regression task
        self.model.roi_heads.box_predictor.cls_score = nn.Linear(self.model.roi_heads.box_predictor.cls_score.in_features, 2)

    def forward(self, x):
        return self.model(x)


In [21]:
class DenseNetRegressor(nn.Module):
    def __init__(self):
        super(DenseNetRegressor, self).__init__()
        # Use pre-trained DenseNet (e.g., DenseNet121) from timm
        self.model = timm.create_model("densenet121", pretrained=True)
        self.model.classifier = nn.Linear(self.model.classifier.in_features, 2)

    def forward(self, x):
        return self.model(x)


In [23]:
# Training Setup
model = DenseNetRegressor().cuda()
criterion = nn.L1Loss()  # MAE Loss
optimizer = optim.AdamW(
    model.parameters(), 
    lr=1e-4,            # Base learning rate
    betas=(0.9, 0.999), # Default betas, can experiment with adjustments
    eps=1e-8,           # Small epsilon to avoid division by zero
    weight_decay=1e-5   # L2 regularization to prevent overfitting
)


# Model Saving Path
best_model_path = "best_model.pth"

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

In [26]:
import torch
from tqdm import tqdm

def train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs=40, patience=5, best_model_path="best_model.pth", device=None):
    
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model.to(device)  # Move model to the appropriate device
    
    best_loss = float("inf")  # Initialize the best loss as infinity
    epochs_no_improve = 0     # Counter for epochs with no improvement
    
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        epoch_loss = 0.0
        
        # Training loop
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
            images, targets = images.to(device), targets.to(device)  # Move to device
            
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            # Accumulate the training loss
            epoch_loss += loss.item()
        
        # Validation loop
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        with torch.no_grad():  # No gradient tracking during validation
            for images, targets in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
                images, targets = images.to(device), targets.to(device)  # Move to device
                outputs = model(images)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
        
        # Calculate average validation loss
        val_loss /= len(valid_loader)
        
        # Print the results for the epoch
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}")
        
        # Save the best model based on validation loss
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0  # Reset counter if validation loss improves
        else:
            epochs_no_improve += 1
        
        # Early stopping: if no improvement in validation loss for 'patience' epochs, stop training
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs with no improvement.")
            break
            
    # Load the best model weights
    model.load_state_dict(torch.load(best_model_path))
    
    return model


In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [28]:
model = train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs=50, patience=5, best_model_path="best_model.pth", device=device)

Epoch 1/50 - Validation: 100%|██████████| 21/21 [01:58<00:00,  5.65s/it]


Epoch 1/50, Train Loss: 2.5882, Val Loss: 2.6639


Epoch 2/50 - Validation: 100%|██████████| 21/21 [01:44<00:00,  4.98s/it]


Epoch 2/50, Train Loss: 2.3305, Val Loss: 2.2183


Epoch 3/50 - Validation: 100%|██████████| 21/21 [01:44<00:00,  5.00s/it]


...

Epoch 14/50 - Validation: 100%|██████████| 21/21 [01:45<00:00,  5.00s/it]


Epoch 14/50, Train Loss: 1.2604, Val Loss: 1.8690


Epoch 15/50 - Validation: 100%|██████████| 21/21 [01:44<00:00,  4.98s/it]

Epoch 15/50, Train Loss: 1.2164, Val Loss: 1.7460
Early stopping triggered after 15 epochs with no improvement.



<ipython-input-26-db3ecd9cafc1>:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


In [29]:
# Load Best Model
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Predict on Validation Set
preds = []
true_vals = []
with torch.no_grad():
    for images, targets in tqdm(valid_loader, desc="Predicting on Validation Set"):
        images = images.cuda()
        outputs = model(images).cpu().numpy()
        preds.append(outputs)
        true_vals.append(targets.numpy())
preds = np.concatenate(preds, axis=0)
true_vals = np.concatenate(true_vals, axis=0)

<ipython-input-29-ce980cad07c1>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))
Predicting on Validation Set: 100%|██████

In [30]:
from sklearn.metrics import mean_absolute_error

# Evaluate using MAE
mae = mean_absolute_error(true_vals, preds)
print(f"Validation MAE: {mae:.4f}")

Validation MAE: 1.6585


In [31]:
# Predict on Test Set
test_df = pd.read_csv("/kaggle/input/lacuna-solar-survey-challenge/Test.csv")

test_df["path"] = "/kaggle/input/lacuna-solar-survey-challenge/images/" + test_df["ID"] + ".jpg"

dataset_test = SolarPanelDataset(test_df, transform=test_transforms, to_train=False)
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False)

test_preds = []
with torch.no_grad():
    for images in tqdm(test_loader, desc="Predicting on Test Set"):
        images = images.cuda()
        outputs = model(images).cpu().numpy()
        test_preds.append(outputs)
test_preds = np.concatenate(test_preds, axis=0)

Predicting on Test Set: 100%|██████████| 35/35 [03:21<00:00,  5.77s/it]


In [33]:
# Create Sample Submission
submission = pd.DataFrame()
submission["ID"] = np.repeat(test_df["ID"].values, 2)
submission["ID"] = submission["ID"] + np.tile(["_boil", "_pan"], len(test_df))
submission["Target"] = test_preds.flatten().clip(0,1000)

# Save Submission
submission.to_csv("lacuna sub1.csv", index=False)
print("Sample submission saved!")

Sample submission saved!
